# Proteomics, Post-translational Modifications, andIntegrative Analyses Reveal MolecularHeterogeneity within Medulloblastoma Subgroups

## SIMILARITY NETWORK FUSION ANALYSIS

### **Abstract**
There is a pressing need to identify therapeutic targets in tumors with low mutation rates such as the malignant pediatric brain tumor medulloblastoma. To address this challenge, we quantitatively profiled global proteomes and phospho-proteomes of 45 medulloblastoma samples. Integrated analyses revealed that tumors with similar RNA expression vary extensively at the post-transcriptional and post-translational levels. We identified distinct pathways associated with two subsets of SHH tumors, and found post-translational modifications of MYC that are associated with poor outcomes in Group 3 tumors. We found kinases associated with subtypes and showed that inhibiting PRKDC sensitizes MYC-driven cells to radiation. Our study shows that proteomics enables a more comprehensive, functional readout, providing a foundation for future therapeutic strategies.

![experimental_design](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6372116/bin/nihms-1503512-f0001.jpg)
**Figure 1:** Summary of data types included in this study, depth of proteomic data types, and cohort composition.

Consensus clustering, principal component analysis (PCA), and t-distributed stochastic neighbor embedding (t-SNE) carried out separately on each data type predominantly revealed the known subgroups: Group 3, Group 4, and SHH (Figures ​(Figures2A,2A, S1-S3). The proteomic data also identified very stable subsets of two known subgroups. Here, we refer to Group 3 clusters as Group 3a (G3a) and Group 3b (G3b), and SHH clusters as SHHa and SHHb.

![subgroups](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6372116/bin/nihms-1503512-f0002.jpg)
**Figure 2:** Comparison of clustering results.

REF: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6372116/

### Objectives
Use **CKG's analytics core** to:

1. Re-analyze the proteomics dataset: 

    - Preprocessing
    - Stratification looking at covariates: Age, Gender, Metastatic status and histology
    - Differential regulation including covariates
    - Functional enrichment

2. Identify patients subgroups 

    - Integration of all omics datasets using Similarity Network Fusion (REF: https://www.nature.com/articles/nmeth.2810).
    - Idenfity clinical and molecular differences between the identifierd subgroups

3. Investigate correlating PTMs on relevant protein complexes

# Loading the Data

In this analysis we use:

- Clinical metadata
- RNA-sequencing 
- Proteomics
- Phosphoproteomics
- Phosphotyrosine proteomics
- Acetylomics

This datasets have been previously formated (samples x proteins) and standardized to UniProt identifiers (data available in */assets*) (see notebook Meduloblastoma Data Processing.ipynb).

In [ ]:
import pandas as pd
import numpy as np

from analytics_core.analytics import analytics
from analytics_core.viz import viz

from graphdb_builder import mapping
from graphdb_connector import connector

from plotly.offline import init_notebook_mode, iplot

%matplotlib inline
init_notebook_mode(connected=True)

In [ ]:
clinical_data = pd.read_excel('assets/samples.xlsx', header=0)
rnaseq_data = pd.read_csv("assets/rnaseq.tsv", sep='\t', header=0)
proteomics_data = pd.read_csv("assets/proteomics.tsv", sep='\t', header=0)
phosphoSTY_data = pd.read_csv("assets/phosphoSTY.tsv", sep='\t', header=0)
phosphoY_data = pd.read_csv("assets/phosphoY.tsv", sep='\t', header=0)
acetylomics_data = pd.read_csv("assets/acetylomics.tsv", sep='\t', header=0)

# 2. Identify patients subgroups 

## Preprocessing:

0. Remove WNT samples (low number of samples)
1. Filter proteins/PTMs/transcipts with more than 80% missing values in all groups
2. Normalize data using median-centered normalization
3. Impute missing values using a mixed model: K-Nearest Neighbors algorithm (KNN) if at least 60% valid values, otherwise Minimum Probability (MinProb) drawing random values from a down-shifted Gaussian distribution
4. Include Protein name in columns using CKG's mapping functionality


#### 0. Remove WNT samples

In [ ]:
proteomics_data = proteomics_data[proteomics_data['group'] != 'WNT']
phosphoSTY_data = phosphoSTY_data[phosphoSTY_data['group'] != 'WNT']
phosphoY_data = phosphoY_data[phosphoY_data['group'] != 'WNT']
acetylomics_data = acetylomics_data[acetylomics_data['group'] != 'WNT']
rnaseq_data = rnaseq_data[rnaseq_data['group'] != 'WNT']

#### 1. Filter proteins with more than 80% missing values in all groups

In [ ]:
valid_proteins = analytics.extract_percentage_missing(proteomics_data, missing_max=0.2, drop_cols=['index'], group='group', how='all')
proteomics_data = proteomics_data[valid_proteins + ['index', 'group']]

valid_sites = analytics.extract_percentage_missing(phosphoSTY_data, missing_max=0.2, drop_cols=['index'], group='group', how='all')
phosphoSTY_data = phosphoSTY_data[valid_sites + ['index', 'group']]

valid_sites = analytics.extract_percentage_missing(phosphoY_data, missing_max=0.2, drop_cols=['index'], group='group', how='all')
phosphoY_data = phosphoY_data[valid_sites + ['index', 'group']]

valid_sites = analytics.extract_percentage_missing(acetylomics_data, missing_max=0.2, drop_cols=['index'], group='group', how='all')
acetylomics_data = acetylomics_data[valid_sites + ['index', 'group']]

valid_transcripts = analytics.extract_percentage_missing(rnaseq_data, missing_max=0.2, drop_cols=['index'], group='group', how='all')
rnaseq_data = rnaseq_data[valid_transcripts + ['index', 'group']]

#### 2. Normalize data using median-centered normalization

In [ ]:
proteomics_data = analytics.normalize_data(proteomics_data, method='median', normalize='samples')

phosphoSTY_data = analytics.normalize_data(phosphoSTY_data, method='median', normalize='samples')

phosphoY_data = analytics.normalize_data(phosphoY_data, method='median', normalize='samples')

acetylomics_data = analytics.normalize_data(acetylomics_data, method='median', normalize='samples')

rnaseq_data = analytics.normalize_data(rnaseq_data, method='median', normalize='samples')

#### 3. Impute missing values using a mixed model

In [ ]:
proteomics_data = analytics.imputation_mixed_norm_KNN(proteomics_data, index_cols=['index','group']).reset_index()
proteomics_data = proteomics_data.rename(columns={'index':'sample'})

phosphoSTY_data = analytics.imputation_mixed_norm_KNN(phosphoSTY_data, index_cols=['index','group']).reset_index()
phosphoSTY_data = phosphoSTY_data.rename(columns={'index':'sample'})

phosphoY_data = analytics.imputation_mixed_norm_KNN(phosphoY_data, index_cols=['index','group']).reset_index()
phosphoY_data = phosphoY_data.rename(columns={'index':'sample'})

acetylomics_data = analytics.imputation_mixed_norm_KNN(acetylomics_data, index_cols=['index','group']).reset_index()
acetylomics_data = acetylomics_data.rename(columns={'index':'sample'})

rnaseq_data = analytics.imputation_mixed_norm_KNN(rnaseq_data, index_cols=['index','group']).reset_index()
rnaseq_data = rnaseq_data.rename(columns={'index':'sample'})

#### 4. Include Protein name in columns

In [ ]:
m = mapping.getMappingFromDatabase(proteomics_data.columns, node="Protein", attribute_from='id', attribute_to='name')
columns = [m[c]+"~"+c if c in m and m[c] is not None else c for c in proteomics_data.columns]
proteomics_data.columns = columns

phospho_prots = [c.split('_')[0] for c in phosphoSTY_data.columns]
m = mapping.getMappingFromDatabase(phospho_prots, node="Protein", attribute_from='id', attribute_to='name')
columns = [m[c.split('_')[0]]+"_"+c.split('_')[1] if c.split('_')[0] in m and m[c.split('_')[0]] is not None else c for c in phosphoSTY_data.columns]
phosphoSTY_data.columns = columns

phospho_prots = [c.split('_')[0] for c in phosphoY_data.columns]
m = mapping.getMappingFromDatabase(phospho_prots, node="Protein", attribute_from='id', attribute_to='name')
columns = [m[c.split('_')[0]]+"_"+c.split('_')[1] if c.split('_')[0] in m and m[c.split('_')[0]] is not None else c for c in phosphoY_data.columns]
phosphoY_data.columns = columns

ace_prots = [c.split('_')[0] for c in acetylomics_data.columns]
m = mapping.getMappingFromDatabase(ace_prots, node="Protein", attribute_from='id', attribute_to='name')
columns = [m[c.split('_')[0]]+"_"+c.split('_')[1] if c.split('_')[0] in m and m[c.split('_')[0]] is not None else c for c in acetylomics_data.columns]
acetylomics_data.columns = columns

In [ ]:
proteomics_data.head()

# Smilarity Network Fusion

![SimilarityNetworkFusion](https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fnmeth.2810/MediaObjects/41592_2014_Article_BFnmeth2810_Fig1_HTML.jpg)

Integration of all omics data generated using Similarity Network Fusion (SNF):

- SNF builds sample similarity networks using each omics data (RNAseq, Proteomics, PhosphoSTY, PhosphoY, Acetylomics) and fuses those networks into a single network that uses to identify clusters

- We use these clusters to identify subgroups of patients

In [ ]:
df_dict = {'rnaseq': rnaseq_data, 'proteomics': proteomics_data, 'phosphoSTY': phosphoSTY_data, 'phosphoY': phosphoY_data, 'acetylomics': acetylomics_data}
feature_df, fused_labels, silhouette_score = analytics.run_snf(df_dict, index=['sample', 'group'], num_clusters=None, distance_metric='euclidean', k_affinity=5, mu_affinity=0.5)

We include the identified clusters into the clinical data (**cluster**)

In [ ]:
common_samples = set() 
for d in df_dict:
    df = df_dict[d]
    df = df.set_index(['sample','group'])
    if len(common_samples) > 1:
        common_samples = common_samples.intersection(df.index)
    else:
        common_samples.update(df.index.tolist())
clinical_data = clinical_data.set_index(['sample','group']).loc[list(common_samples)]
clinical_data['cluster'] = fused_labels.values
clinical_data["cluster"] = clinical_data["cluster"].astype('str')

In [ ]:
clinical_data.head()

# Comparing SNF Clusters to Known Clinical Groups

We compare the identified 4 clusters with the known clinical groups (GR3, GR4, SHH) and the ones described by Archer et al. (G3a, G3b, G4, SHHa, SHHb).

- SNF perfectly groups into one cluster (cluster 3) GR3 samples
- Cluster 1 contains the majority of GR4 samples. Some of the samples in this group are instead clustered with GR3 samples
- Cluster 2 groups previously described as a subset of SHH samples SHHa and cluster 4 contains the other subgroup SHHb

In [ ]:
clinical_data.groupby(['Proteome-based ', 'group', 'cluster']).count()['Histology']

In [ ]:
df = proteomics_data.set_index(['sample', 'group']).join(clinical_data)
df = df.dropna(subset=['cluster'])

In [ ]:
pca, args = analytics.run_pca(df, drop_cols=['Proteome-based', 'Age At Diagnosis (years)', 'group', 'Metastatic  Status', 'Histology', 'Gender'], group='cluster', annotation_cols=['sample'], components=2)

In [ ]:
args.update({'title':"PCA Medulloblastoma by SNF cluster", 'loadings':25, 'factor': 50})
figure = viz.get_pca_plot(pca, identifier='pca_plot', args=args)
iplot(figure.figure)
viz.save_DASH_plot(figure, name="PCA_Medulloblastoma_gender", plot_format='png', directory='.')

## Selecting Most Informative Features

SNF outputs Mutual Information scores (MIscore) for all the features included in the analysis. This score can be used to rank features and their contribution to the 4 clusters found.

- The distribution of scores can be used to rank the most informative technologies.
- We use the best ranking features (MIscore >=0.5) to project the samples using PCA analysis and to find correlations among these features and how their expression profiles link to known clinical groups.

In [ ]:
feature_df = feature_df.sort_values(by='MIscore', ascending=False)

In [ ]:
feature_df.describe()

In [ ]:
feature_df.head(n=10)

### Selected Features

In [ ]:
selected_features = feature_df[feature_df['MIscore']>0.5].index.tolist()

In [ ]:
selected_features

## PCA with SNF Clusters

We use the selected features to project the samples and color them according to their SNF cluster. We symbols to represent the clinical group they were annotated to.

In [ ]:
all_df = pd.DataFrame(index=['sample', 'group'])
for d in df_dict:
    df = df_dict[d]
    df = df.set_index(['sample','group'])
    if all_df.empty:
        all_df = df.loc[list(common_samples)]
    else:
        all_df = all_df.join(df.loc[list(common_samples)], rsuffix='Y')
all_df = all_df[selected_features]

In [ ]:
df = all_df.join(clinical_data[['cluster', 'Proteome-based ']].dropna()).sort_values(by='cluster')
df = df.reset_index()

In [ ]:
pca, args = analytics.run_pca(df, drop_cols=['group'], group='cluster', annotation_cols=['sample', 'Proteome-based '], components=2)
args.update({'title':"PCA Medulloblastoma by SNF cluster", 'loadings':25, 'factor': 20, 'symbol':'Proteome-based ', 'hovering_cols':['sample', 'Proteome-based ']})
figure = viz.get_pca_plot(pca, identifier='pca_plot', args=args)
iplot(figure.figure)
viz.save_DASH_plot(figure, name="PCA_Medulloblastoma_SNF_clusters_based_on_selected_features", plot_format='png', directory='.')

## Correlation Network and Expression Profiles

We run correlation analysis to identify correlations among the selected features. The correlation network can then be used to identify communities of correlating features (Affinity Propagation algorithm) and these communities can be linked to especific behavior in the clinical groups (SNF clusters).

In [ ]:
corr = analytics.run_correlation(df, subject='sample')

In [ ]:
corr.head()

In [ ]:
net = viz.get_network(corr, identifier='corr_net', args={'source':'node1', 'target':'node2', 
                                                         'values':'weight', 'title':'Correlation network', 
                                                         'color_weight':True, 'cutoff':0.6, 
                                                         'cutoff_abs':True, 'format':'edgelist', 
                                                         'communities_algorithm': 'affinity_propagation'})

In [ ]:
viz.visualize_notebook_network(net['notebook'])

In [ ]:
nodes_table, edges_table = net['net_tables']
for i, c in nodes_table.groupby('cluster'):
    selected = c['index'].tolist()
    
    plot = viz.get_parallel_plot(df.set_index("cluster")[selected].reset_index(), 
                      identifier='parallel_plot', 
                      args={"color":c['color'].tolist()[0], "title": "Profile features: {}".format(', '.join(selected)), "zscore":True, "group":"cluster"})
    viz.save_DASH_plot(plot, name="profile_plot_{}".format('_'.join(selected)), plot_format='png', directory='.')
    iplot(plot.figure)

## Most Informative Technology

We look at the distribution of Mutual Information scores to identify what technology is the most informative.

In [ ]:
figure = viz.get_distplot(feature_df, identifier='distributions', args={'x': 'dataset', 'y': 'MIscore', 'group': 'dataset'})
iplot(figure[0].figure)

In [ ]:
figure = viz.get_violinplot(feature_df, identifier='distributions', args={'x': 'dataset', 'y': 'MIscore', 'group':'dataset'})
iplot(figure[0].figure)

# Differential Regulation Across SNF Clusters

In [ ]:
clinical_data.head()

In [ ]:
proteomics_data = proteomics_data.set_index(['sample', 'group']).join(clinical_data).reset_index()
proteomics_data = proteomics_data.drop('Proteome-based ', axis=1)
proteomics_data.loc[proteomics_data['Metastatic  Status'].isna(), 'Metastatic  Status'] = 'M3'

In [ ]:
proteomics_data = proteomics_data.dropna()

In [ ]:
proteomics_data.head()

In [ ]:
pro_anova_result = analytics.run_ancova(proteomics_data, covariates=['Age At Diagnosis (years)', 'Gender', 'Histology', 'Metastatic  Status'], drop_cols=['sample', 'group'], group='cluster', subject=None, permutations=0)

In [ ]:
anova_significant = pro_anova_result[pro_anova_result.rejected & (pro_anova_result['posthoc padj'] <= 0.01)]

In [ ]:
anova_significant['identifier'].unique().shape

In [ ]:
volcano_plot = viz.run_volcano(pro_anova_result, identifier='proteomics_volcanos', 
            args={'alpha':0.01, 'fc':2, 'num_annotations': 100,
                  'colorscale':'Blues', 'showscale': False, 
                  'marker_size':8, 'x_title':'log2FC', 'y_title':'-log10(pvalue)'})
i = 0
for plot in volcano_plot:
    iplot(plot.figure)
    viz.save_DASH_plot(plot, name="volcano_plot_"+str(i), plot_format='png', directory='.')
    i += 1

## Functional Enrichment Significant Proteins SNF Clusters

In [ ]:
go_terms_query = "MATCH (p:Protein)-[]-(bp:Biological_process) WHERE (p.name+'~'+p.id) IN {} RETURN DISTINCT (p.name+'~'+p.id) AS identifier,bp.name AS annotation"
go_terms_query = go_terms_query.format(proteomics_data.columns.tolist())
annotation = connector.run_query(go_terms_query)

In [ ]:
enrichment_results = analytics.run_up_down_regulation_enrichment(pro_anova_result, annotation, identifier='identifier', groups=['group1', 'group2'], annotation_col='annotation', reject_col='rejected', group_col='group', method='fisher', correction='fdr_bh', alpha=0.01, lfc_cutoff=1)

In [ ]:
figures = viz.get_enrichment_plots(enrichment_results, identifier='enrichment', args={'width':2200})
i = 0
for fig in figures:
    iplot(fig.figure)
    viz.save_DASH_plot(fig, name="enrichment_"+str(i), plot_format='png', directory='.')
    i += 1

# References

1. https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6372116/
2. https://www.nature.com/articles/nmeth.2810